# FedNCF Baseline — H&M Sampled Dataset

Neural Collaborative Filtering with federated training on the H&M sampled data.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


## 1. Load Data


In [4]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
TARGET_USERS     = 1000
MIN_INTERACTIONS = 20
N_QUANTILES      = 4
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. Run sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded : {cache_tag}")
print(f"Users  : {n_users} | Items : {n_items}")
print(f"Train  : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")


Loaded : u1000_m20_s42
Users  : 948 | Items : 7418
Train  : 38708 | Val : 9677 | Test : 15696


## 2. Build Rating Matrices


In [6]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["customer_id"].values,  dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating).to(device)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating).to(device)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating).to(device)

print(f"Train : {(train_matrix != -1).sum().item()} observed")
print(f"Val   : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  : {(test_matrix  != -1).sum().item()} observed")


Train : 38708 observed
Val   : 9677 observed
Test  : 15696 observed


## 3. Model & Loss


In [8]:
class Fed_NCF(nn.Module):
    def __init__(self, user_num, item_num, factor_num):
        super().__init__()
        self.embed_user = nn.Embedding(user_num, factor_num)
        self.embed_item = nn.Embedding(item_num, factor_num)
        self.predict    = nn.Linear(factor_num, 1)
        nn.init.normal_(self.embed_user.weight, std=0.01)
        nn.init.normal_(self.embed_item.weight, std=0.01)

    def forward(self, user, item):
        u = self.embed_user(user)                          # (n_users, K)
        v = self.embed_item(item)                          # (n_items, K)
        x = torch.einsum('ik,jk->ijk', [u, v])            # (n_users, n_items, K)
        return torch.sigmoid(self.predict(x)).squeeze(-1) # (n_users, n_items)


class NCFLoss(nn.Module):
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam

    def forward(self, matrix, predicted, model):
        mask  = (matrix != -1).float()
        error = torch.sum(((matrix - predicted) ** 2) * mask)
        reg   = self.lam * sum(p.norm() for p in model.parameters())
        return error + reg


def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0: return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())

print("Fed_NCF, NCFLoss, compute_rmse defined.")


Fed_NCF, NCFLoss, compute_rmse defined.


## 4. Parameter Tuning -- Grid Search

Run **once** after building matrices. Writes `lr`, `lam`, `latent_vectors` in-place.


In [10]:
# ── Grid ─────────────────────────────────────────────────────────────────────
LR_GRID     = [0.001, 0.01,  0.05]
LAM_GRID    = [0.01,  0.1,   0.3 ]
LATENT_GRID = [10,    20,    40  ]
TUNE_EPOCHS = 30
TUNE_PAT    = 5

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)
grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    _model = Fed_NCF(n_users, n_items, _K).to(device)
    _loss_fn = NCFLoss(lam=_lam)
    _opt = torch.optim.Adam(_model.parameters(), lr=_lr)
    _user = torch.arange(n_users, device=device)
    _item = torch.arange(n_items, device=device)
    _best_val, _wait = float('inf'), 0
    for _ep in range(TUNE_EPOCHS):
        _model.train()
        _opt.zero_grad()
        _pred = _model(_user, _item)
        _loss = _loss_fn(train_matrix, _pred, _model)
        _loss.backward()
        _opt.step()
        _model.eval()
        with torch.no_grad():
            _v = compute_rmse(val_matrix, _model(_user, _item), min_rating, max_rating)
        if _v < _best_val: _best_val, _wait = _v, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT: break
    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")

best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])
print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combinations x up to 30 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    10          8.3963 <--
  2   0.0010   0.010    20          7.9517 <--
  3   0.0010   0.010    40          8.6716
  4   0.0010   0.100    10          8.3957
  5   0.0010   0.100    20          7.9506 <--
  6   0.0010   0.100    40          8.6698
  7   0.0010   0.300    10          8.3952
  8   0.0010   0.300    20          7.9499 <--
  9   0.0010   0.300    40          8.6693
 10   0.0100   0.010    10          6.5112 <--
 11   0.0100   0.010    20          5.8627 <--
 12   0.0100   0.010    40          5.0546 <--
 13   0.0100   0.100    10          6.2558
 14   0.0100   0.100    20          5.3988
 15   0.0100   0.100    40          4.4574 <--
 16   0.0100   0.300    10          6.3234
 17   0.0100   0.300    20          5.2226
 18   0.0100   0.300    40          4.2638 <--
 19   0.0500   0.010    10          2.1443 <--
 20   0.05

## 5. Training


In [12]:
if 'latent_vectors' not in dir(): latent_vectors = 20
if 'lr'  not in dir(): lr  = 0.01
if 'lam' not in dir(): lam = 0.1
num_epoch = 100
patience  = 10

torch.manual_seed(0)
model    = Fed_NCF(n_users, n_items, latent_vectors).to(device)
loss_fn  = NCFLoss(lam=lam)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
user_idx = torch.arange(n_users, device=device)
item_idx = torch.arange(n_items, device=device)

best_val_rmse, patience_count, best_state = float('inf'), 0, None

for epoch in range(num_epoch):
    model.train()
    optimizer.zero_grad()
    pred = model(user_idx, item_idx)
    loss = loss_fn(train_matrix, pred, model)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_eval  = model(user_idx, item_idx)
        train_rmse = compute_rmse(train_matrix, pred_eval, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_eval, min_rating, max_rating)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {loss.item():.3f} "
              f"| train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)
print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Epoch   0 | loss: 7033.716 | train RMSE: 7.8674 | val RMSE: 7.8599
Epoch  10 | loss: 2122.791 | train RMSE: 3.5926 | val RMSE: 4.1498
Epoch  20 | loss: 197.861 | train RMSE: 0.9789 | val RMSE: 1.5403
Epoch  30 | loss: 216.307 | train RMSE: 0.9678 | val RMSE: 1.4805
Epoch  40 | loss: 216.157 | train RMSE: 0.9546 | val RMSE: 1.4800
Early stopping at epoch 44.

Best val RMSE: 1.4696


## 6. Test Evaluation


In [14]:
model.eval()
with torch.no_grad():
    pred_test = model(user_idx, item_idx)
    test_rmse = compute_rmse(test_matrix, pred_test, min_rating, max_rating)
    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test   * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")


Test RMSE : 1.3734
Test MAE  : 0.5691
